In [1]:
import random
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Literal

import pandas as pd
from openai import OpenAI
from pydantic import BaseModel, field_validator
from tqdm.auto import tqdm

In [2]:
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL_NAME = "llama3:instruct"   

N_CANDIDATES = 2        # mismo prompt ejecutado N veces (en paralelo), el juez elige
GEN_TEMPERATURE = 0.8
GEN_MAX_TOKENS = 200    
GEN_TRIES = 2

JUDGE_PASSES = 1        # 1 = rapido (orden aleatorio) | 2 = doble pasada con orden invertido
JUDGE_REASON = False    # True = el juez escribe una razon corta (+3 s)
JUDGE_MAX_TOKENS = 60 if JUDGE_REASON else 10

WORKERS = 2             # peliculas en paralelo en run_dataset (WORKERS x N_CANDIDATES <= OLLAMA_NUM_PARALLEL)
SEED = 42
DEBUG = True

_use_schema = True      # se desactiva solo si Ollama rechaza json_schema

In [3]:
# Encuadres (los elige el codigo, con pesos por genero, para que haya variedad)
# nombre: (instruccion para el LLM, tag para Stable Diffusion)
FRAMINGS = {
    "close portrait":   ("close-up portrait of the main character, face and costume fill the frame",
                         "close-up portrait"),
    "medium character": ("medium shot of the main character, environment visible behind",
                         "medium shot"),
    "wide environment": ("wide establishing shot, huge environment, characters tiny in the frame",
                         "extreme wide shot, vast environment, tiny figures"),
    "silhouette":       ("lone figure seen as a dark silhouette against a large dramatic background",
                         "silhouette against dramatic background, wide shot"),
    "symbolic object":  ("one symbolic object or prop centered, minimal composition, no visible face",
                         "centered object, minimal symmetrical composition"),
    "duo":              ("two characters facing each other or side by side",
                         "two characters, medium shot"),
}

# pesos editables
FRAMING_WEIGHTS = {
    "terror":          {"close portrait": 2, "medium character": 2, "wide environment": 2, "silhouette": 3, "symbolic object": 3, "duo": 1},
    "comedia":         {"close portrait": 2, "medium character": 4, "wide environment": 1, "silhouette": 0, "symbolic object": 1, "duo": 3},
    "ciencia_ficcion": {"close portrait": 2, "medium character": 2, "wide environment": 4, "silhouette": 3, "symbolic object": 2, "duo": 1},
    "romance":         {"close portrait": 3, "medium character": 3, "wide environment": 1, "silhouette": 2, "symbolic object": 1, "duo": 5},
    "animacion":       {"close portrait": 2, "medium character": 4, "wide environment": 2, "silhouette": 0, "symbolic object": 1, "duo": 2},
    "accion":          {"close portrait": 2, "medium character": 4, "wide environment": 3, "silhouette": 3, "symbolic object": 1, "duo": 1},
}

# iluminacion + paleta + estilo por genero (el LLM no los escribe - menos tokens y mas consistencia)
GENRE_LOOKS = {
    "terror": [
        "low-key lighting, heavy shadows, desaturated cold palette, film grain",
        "single harsh light source, deep black shadows, sickly green and grey palette, fog",
        "moonlit blue darkness, red accent light, high contrast, grainy cinematic still",
    ],
    "comedia": [
        "bright high-key lighting, saturated colors, clean studio look",
        "sunny soft daylight, pastel and primary colors, playful vibrant look",
        "colorful gradient backdrop, even lighting, bold saturated palette",
    ],
    "ciencia_ficcion": [
        "teal and orange palette, glowing lights, anamorphic lens flare, cinematic still",
        "cold blue and white palette, volumetric light beams, sleek futuristic look",
        "neon magenta and cyan glow, dark atmosphere, high contrast, cinematic still",
    ],
    "romance": [
        "warm golden hour light, soft focus, pastel warm tones",
        "soft diffused window light, creamy pink and beige palette, shallow depth of field",
        "twilight blue and warm orange glow, bokeh lights, dreamy soft focus",
    ],
    "animacion": [
        "stylized 3d render, vibrant colors, soft studio lighting",
        "bright saturated palette, painterly textures, soft ambient light, whimsical",
        "stylized 3d render, warm sunlight, rich colors, expressive characters",
    ],
    "accion": [
        "dramatic backlight, orange and teal contrast, dust and sparks, cinematic still",
        "harsh midday sun, desaturated earthy palette, gritty film grain",
        "fiery orange glow, smoke and embers, high contrast, dynamic look",
    ],
}

GENRE_EN = {
    "terror": "horror", "comedia": "comedy", "ciencia_ficcion": "science fiction",
    "romance": "romance", "animacion": "animation", "accion": "action",
}

In [4]:
# Prompts del LLM
SYSTEM_GENERATOR = """You write the content tags of a movie poster image prompt for Stable Diffusion 1.5.
Return JSON with these fields, each 2-7 words, lowercase English, concrete things a camera can see:
subject, clothing_props, pose_expression, setting, time_weather

Rules:
- Be faithful to the synopsis: use its characters, objects and places. Do not invent another story.
- Follow the required framing (a wide shot means small figures and a big environment).
- No names of people, characters or the movie title. No abstract words (tension, love, mystery).
- Be specific: never "man in suit" or "beautiful landscape".
- Never mention text, title, poster, logo or watermark.

Example: {"subject": "lone man with short black hair", "clothing_props": "long black leather coat, round dark sunglasses", "pose_expression": "standing still, calm stare", "setting": "rain-soaked server room, hanging cables", "time_weather": "night, indoors"}"""

_JUDGE_FORMAT = (
    'Return JSON: {"reason": "<max 12 words>", "winner": "1" or "2"}. Write the reason first.'
    if JUDGE_REASON else
    'Return JSON: {"winner": "1" or "2"}.'
)
SYSTEM_JUDGE = f"""You compare two image prompt drafts for the same movie poster.
Pick the draft that is more faithful to the synopsis (right characters, objects, setting) and more specific and visual.
Ignore the order of the drafts.
{_JUDGE_FORMAT}"""

# Esquemas
CONTENT_FIELDS = ["subject", "clothing_props", "pose_expression", "setting", "time_weather"]

In [5]:
class Content(BaseModel):
    subject: str
    clothing_props: str
    pose_expression: str
    setting: str
    time_weather: str


class _WinnerMixin(BaseModel):
    winner: Literal["1", "2"]

    @field_validator("winner", mode="before")
    @classmethod
    def _norm(cls, v):
        return re.sub(r"\D", "", str(v))[:1]  


class VerdictReason(_WinnerMixin):
    reason: str          # primero la razon, luego el veredicto
    winner: Literal["1", "2"]


class VerdictOnly(_WinnerMixin):
    pass


Verdict = VerdictReason if JUDGE_REASON else VerdictOnly

In [6]:
# Llamada a ollama
def _clean(txt: str) -> str:
    txt = txt.strip()
    txt = re.sub(r"^```(?:json)?\s*", "", txt, flags=re.IGNORECASE)
    return re.sub(r"\s*```$", "", txt)


def chat_json(system, user, schema, temperature, max_tokens):
    global _use_schema
    kwargs = dict(
        model=MODEL_NAME,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
    )
    if _use_schema:
        try:
            r = client.chat.completions.create(
                **kwargs,
                response_format={
                    "type": "json_schema",
                    "json_schema": {"name": schema.__name__, "schema": schema.model_json_schema()},
                },
            )
        except Exception:
            _use_schema = False   # esta version de Ollama no lo soporta: caemos a json_object
            r = client.chat.completions.create(**kwargs, response_format={"type": "json_object"})
    else:
        r = client.chat.completions.create(**kwargs, response_format={"type": "json_object"})
    return schema.model_validate_json(_clean(r.choices[0].message.content))

In [7]:
# Generador
BANNED = re.compile(r"\b(poster|title|text|logo|typography|watermark|caption|subtitle)\b", re.I)


def content_text(c: Content) -> str:
    parts = [getattr(c, f).strip().strip(",.").lower() for f in CONTENT_FIELDS]
    return ", ".join(p for p in parts if p)


def prompt_issue(c: Content, title: str):
    """Motivo por el que el contenido no vale, o None si es valido."""
    for f in CONTENT_FIELDS:
        if len(getattr(c, f).strip()) < 2:
            return f"campo vacio: {f}"
    text = content_text(c)
    if len(text.split()) > 60:
        return f"{len(text.split())} palabras (max 60)"
    if "_" in text:
        return "contiene guion bajo"
    m = BANNED.search(text)
    if m:
        return f"palabra prohibida: {m.group(0)!r}"
    if len(title) > 3 and re.search(r"\b" + re.escape(title.lower()) + r"\b", text.lower()):
        return "contiene el titulo"
    return None


def _gen_one(user, title, idx):
    for attempt in range(1, GEN_TRIES + 1):
        try:
            c = chat_json(SYSTEM_GENERATOR, user, Content, GEN_TEMPERATURE, GEN_MAX_TOKENS)
        except Exception as e:
            if DEBUG:
                print(f"  [cand {idx} intento {attempt}] error: {type(e).__name__}: {str(e)[:200]}")
            continue
        issue = prompt_issue(c, title)
        if issue is None:
            return c
        if DEBUG:
            print(f"  [cand {idx} intento {attempt}] rechazado: {issue} | {content_text(c)}")
    return None

In [8]:
# Juez
def judge(overview, genre, texts, rng):
    """texts: contenido de los 2 candidatos. Devuelve (indice ganador 0/1, acuerdo, razon)."""
    orders = [False, True] if JUDGE_PASSES == 2 else [rng.random() < 0.5]
    votes = []
    for swap in orders:
        first, second = (texts[1], texts[0]) if swap else (texts[0], texts[1])
        user = (
            f"Genre: {GENRE_EN.get(genre, genre)}\nSynopsis: {overview}\n\n"
            f"Draft 1: {first}\n\nDraft 2: {second}"
        )
        v = chat_json(SYSTEM_JUDGE, user, Verdict, 0.0, JUDGE_MAX_TOKENS)
        picked_first = v.winner == "1"
        idx = (1 if picked_first else 0) if swap else (0 if picked_first else 1)
        votes.append((idx, getattr(v, "reason", "")))

    if len(votes) == 1:
        return votes[0][0], "single", votes[0][1]
    if votes[0][0] == votes[1][0]:
        return votes[0][0], "consistent", votes[0][1]
    idx = rng.choice([0, 1])
    return idx, "tie", next(r for i, r in votes if i == idx)

In [9]:
# Pipeline por pelicula
def sample_framing(genre, rng):
    w = FRAMING_WEIGHTS.get(genre, {})
    names = list(FRAMINGS)
    return rng.choices(names, weights=[w.get(n, 1) for n in names])[0]


def procesar_pelicula(title, overview, genre, year, key=None):
    rng = random.Random(f"{SEED}-{key if key is not None else title}")   # reproducible y seguro con hilos
    framing = sample_framing(genre, rng)
    instruction, sd_tag = FRAMINGS[framing]
    look = rng.choice(GENRE_LOOKS.get(genre, [""]))

    user = (
        f"Synopsis: {overview}\n"
        f"Genre: {GENRE_EN.get(genre, genre)} ({year})\n"
        f"Required framing: {instruction}"
    )

    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=N_CANDIDATES) as pool:
        cands = [c for c in pool.map(lambda i: _gen_one(user, title, i), range(N_CANDIDATES)) if c]
    t1 = time.perf_counter()
    if not cands:
        return None

    texts = [content_text(c) for c in cands]
    if len(texts) == 1:
        win, agreement, reason = 0, "single_candidate", ""
    else:
        win, agreement, reason = judge(overview, genre, texts, rng)
    t2 = time.perf_counter()

    full = [", ".join(x for x in (sd_tag, t, look) if x) for t in texts]
    return {
        "framing": framing,
        "prompt_a": full[0],
        "prompt_b": full[1] if len(full) > 1 else "",
        "winner": "AB"[win],
        "judge_agreement": agreement,
        "judge_reason": reason,
        "diffusion_prompt": full[win],
        "t_gen": round(t1 - t0, 2),
        "t_judge": round(t2 - t1, 2),
    }

In [10]:
# Pruebas
PELICULAS_TEST = [
    {"title": "Inception", "genre": "ciencia_ficcion", "year": "2010",
     "overview": "A skilled thief who steals secrets from people's dreams is offered a final job: planting an idea inside the mind of a powerful executive."},
    {"title": "Get Out", "genre": "terror", "year": "2017",
     "overview": "A young man visits his girlfriend's wealthy family in a secluded country estate, where the polite behavior of the hosts slowly reveals something sinister."},
    {"title": "La La Land", "genre": "romance", "year": "2016",
     "overview": "A jazz pianist and an aspiring actress fall in love in Los Angeles while chasing their careers and struggling to keep their relationship alive."},
    {"title": "Toy Story 3", "genre": "animacion", "year": "2010",
     "overview": "As their owner prepares to leave for college, a group of toys end up by mistake in a day care center and must find a way back home."},
    {"title": "Mad Max: Fury Road", "genre": "accion", "year": "2015",
     "overview": "In a desert wasteland ruled by a tyrant, a drifter and a rebel warrior flee across the sands in an armored war rig with a band of escaped prisoners."},
    {"title": "The Grand Budapest Hotel", "genre": "comedia", "year": "2014",
     "overview": "A legendary concierge of a famous European hotel and his young lobby boy get tangled in the theft of a priceless painting and a battle over a family fortune."},
]


def run_test(peliculas=PELICULAS_TEST):
    filas = []
    for peli in tqdm(peliculas, desc="Evaluando sinopsis"):
        res = procesar_pelicula(peli["title"], peli["overview"], peli["genre"], peli["year"])
        print(f"\n {peli['title']} ({peli['genre']})")
        if res is None:
            print("  no se obtuvo un prompt valido")
            continue
        print(f"  • Encuadre: {res['framing']} | ganador {res['winner']} ({res['judge_agreement']}) "
              f"| gen {res['t_gen']}s | juez {res['t_judge']}s")
        if res["judge_reason"]:
            print(f"  • Razon: {res['judge_reason']}")
        print(f"  • PROMPT: {res['diffusion_prompt']}")
        filas.append(res)

    if filas:
        df = pd.DataFrame(filas)
        total = df.t_gen.sum() + df.t_judge.sum()
        print("\n" + "=" * 50)
        print(f"Tiempo medio por peli: {total / len(df):.1f}s (gen {df.t_gen.mean():.1f}s + juez {df.t_judge.mean():.1f}s)")
        print(f"Tiempo del juez: {df.t_judge.sum() / total:.0%} del total")
        print("Encuadres:", df.framing.value_counts().to_dict())
        print("Ganadores:", df.winner.value_counts().to_dict())
    return filas



In [11]:
run_test()

Evaluando sinopsis:   0%|          | 0/6 [00:00<?, ?it/s]


 Inception (ciencia_ficcion)
  • Encuadre: wide environment | ganador A (single) | gen 7.65s | juez 1.11s
  • PROMPT: extreme wide shot, vast environment, tiny figures, small figure with dark hair tucked under hat, tattered gray trenchcoat, black gloves, leather belt, crouched low, intent stare at floor, abandoned factory complex, rusty metal beams, dusky evening, overcast sky, cold blue and white palette, volumetric light beams, sleek futuristic look

 Get Out (terror)
  • Encuadre: close portrait | ganador A (single) | gen 7.39s | juez 1.16s
  • PROMPT: close-up portrait, young man with messy light hair, dark suit, white shirt, black tie, nervous smile, alert eyes, dimly lit country estate hallway, ornate chandelier, evening, dim, moonlit blue darkness, red accent light, high contrast, grainy cinematic still

 La La Land (romance)
  • Encuadre: duo | ganador A (single) | gen 8.53s | juez 1.16s
  • PROMPT: two characters, medium shot, young woman with curly brown hair, man with messy

[{'framing': 'wide environment',
  'prompt_a': 'extreme wide shot, vast environment, tiny figures, small figure with dark hair tucked under hat, tattered gray trenchcoat, black gloves, leather belt, crouched low, intent stare at floor, abandoned factory complex, rusty metal beams, dusky evening, overcast sky, cold blue and white palette, volumetric light beams, sleek futuristic look',
  'prompt_b': 'extreme wide shot, vast environment, tiny figures, slender figure with silver hair, rugged black jumpsuit, leather gloves, leaning against wall, thoughtful gaze, futuristic cityscape at dusk, towering skyscrapers, sunset, hazy atmosphere, cold blue and white palette, volumetric light beams, sleek futuristic look',
  'winner': 'A',
  'judge_agreement': 'single',
  'judge_reason': '',
  'diffusion_prompt': 'extreme wide shot, vast environment, tiny figures, small figure with dark hair tucked under hat, tattered gray trenchcoat, black gloves, leather belt, crouched low, intent stare at floor, 

### Adaptación para data_v2/metadata.csv

In [12]:
"""# Full dataset
def run_dataset(
    split="holdout_gen",      # None = todas las particiones
    limit=None,
    input_csv="data_v2/metadata.csv",
    output_csv="data_v2/metadata_prompts_local.csv",
):
    df = pd.read_csv(input_csv)
    if split:
        df = df[df["split"] == split]
    out = Path(output_csv)
    if out.exists():
        hechas = set(pd.read_csv(out)["tmdb_id"])
        df = df[~df["tmdb_id"].isin(hechas)]
        print(f"Reanudando: quedan {len(df)} peliculas")
    if limit:
        df = df.head(limit)
    rows = df.to_dict("records")

    def work(row):
        return row, procesar_pelicula(row["title"], row["overview"], row["genre"], row["year"], key=row["tmdb_id"])

    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futures = [pool.submit(work, r) for r in rows]
        try:
            for f in tqdm(as_completed(futures), total=len(futures), desc="Generando prompts"):
                row, res = f.result()
                if res is None:
                    print(f"  no se obtuvo un prompt valido: {row['title']}")
                    continue
                pd.DataFrame([{**row, **res}]).to_csv(
                    out, mode="a", header=not out.exists(), index=False, encoding="utf-8"
                )
        except KeyboardInterrupt:
            pool.shutdown(wait=False, cancel_futures=True)
            print("Interrumpido: lo ya guardado se reanuda al relanzar.")
            raise
    print(f"Listo: {output_csv}")


if __name__ == "__main__":
    run_test()"""

'# Full dataset\ndef run_dataset(\n    split="holdout_gen",      # None = todas las particiones\n    limit=None,\n    input_csv="data_v2/metadata.csv",\n    output_csv="data_v2/metadata_prompts_local.csv",\n):\n    df = pd.read_csv(input_csv)\n    if split:\n        df = df[df["split"] == split]\n    out = Path(output_csv)\n    if out.exists():\n        hechas = set(pd.read_csv(out)["tmdb_id"])\n        df = df[~df["tmdb_id"].isin(hechas)]\n        print(f"Reanudando: quedan {len(df)} peliculas")\n    if limit:\n        df = df.head(limit)\n    rows = df.to_dict("records")\n\n    def work(row):\n        return row, procesar_pelicula(row["title"], row["overview"], row["genre"], row["year"], key=row["tmdb_id"])\n\n    with ThreadPoolExecutor(max_workers=WORKERS) as pool:\n        futures = [pool.submit(work, r) for r in rows]\n        try:\n            for f in tqdm(as_completed(futures), total=len(futures), desc="Generando prompts"):\n                row, res = f.result()\n             